# Data Description
- Background: Unit 1.7 補充資料驗證：臺北捷運出入口。提供機關：臺北大眾捷運股份有限公司；官方來源：https://data.taipei/dataset/detail?id=cfa4778c-62c1-497b-b704-756231de348b。原始來源對應與官方最新資源bytes完全相同為不同命題；除歷史界內包已逐byte雜湊對應外，未重新下載認證其他最新資源。
- date received: 學生於本單元提供；實際下載／收到日期未提供。首次本機檢視2026-11-26，不將今日當下載日。
- Path to data file: 官方來源及版本如上；精確本機容器如下，解壓至data/temp僅為衍生暫存。
  - `data/raw/臺北捷運車站出入口座標_1150804.csv`；17,633 bytes；SHA-256 `134d8488c240fb23e04535db6a94da73006f890c83617eeedaa9b26c9f120015`。
- Unit of observation: 捷運出入口位置紀錄，與車站分開，含臺北市外點。
- Sample period: 1150804檔名及更新快照，無逐出入口設置／撤除日期；發布／更新：資源2026-08-04。ZIP時間、檔名時間、觀測／設置時間分開，不由檔名猜全年存量。
- Known issues: 固定設施快照，不證明研究年存在；站名與入口分開、保留跨市點，直線緩衝非實際步行可及性。缺失、重複、坐標及CRS由下方實讀輸出；沒有完整性證據的0僅為本檔未觀測。授權：政府資料開放授權條款第1版；依官方頁，來源與版本須保留。
- Definition for each variable: 下方逐欄型態、缺失、觀測域與定義表。已知官方欄意與本次診斷欄分開；未獲欄位碼表支持者明記未知，不依名稱猜設施單位。
- Data path: 上列每個獨立原始容器；實際讀取成員列於Summary Stats。
- Role: raw。
- Verification status: 以本Notebook乾淨kernel實際輸出與執行metadata為準；不認證已清理。
- SHA-256: 上列每檔固定綁定，載入前逐項重核。
- Next processing notebook: 本單元僅診斷，Chapter 2尚未建立；[驗證一](../code/taipei_supplementary_accident_boundary_validation.ipynb)、[驗證二](../code/taipei_transportation_infrastructure_validation.ipynb)。

Author: 詹明翰、謝子詮、曾翊涵；Created At / Last Modified At: 2026-11-26。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。共同作者與職務不代表已完成個別工作。本輪由Codex協助執行，成員驗收與審查尚待完成。


In [1]:
from pathlib import Path
import sys, hashlib, json, zipfile
import pandas as pd
from IPython.display import display, Markdown
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate/'AGENTS.md').is_file() and (candidate/'templates').is_dir():
        PROJECT_ROOT=candidate;break
else:raise FileNotFoundError('請從本研究工作包內執行')
def fingerprint(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_config.json')=='7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7'
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_support.py')=='1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
from unit17_validation_support import Context, show, safe_text
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',160)
ctx=Context(PROJECT_ROOT,'7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7')
print('Python:',sys.version,'Config:','7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7','Helper:','1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080')


Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)] Config: 7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7 Helper: 1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080


## Change for Your Own Data

使用已核對精確路徑及固定SHA-256；不選第一個glob或最新檔。CP950非法bytes只作可逆診斷；大檔分批讀取、OSM僅研究區2km bbox，保留全國母體數。

In [2]:
FAMILY='mrt'
INPUTS=ctx.family_items(FAMILY)
for item in INPUTS:ctx.file(item['path'])
frames=ctx.load_family(FAMILY)

## Summary

每個容器、實際成員與載入範圍分列；沒有將暫存解壓檔當原始下載資料。

In [3]:
show('固定輸入身分',pd.DataFrame([{k:v for k,v in r.items() if k not in ['members','encoding_checks']} for r in INPUTS]))
show('讀取範圍與CRS／編碼',pd.DataFrame(ctx.summaries[FAMILY]))

**固定輸入身分**

,family,path,bytes,sha256,status
0,mrt,data/raw/臺北捷運車站出入口座標_1150804.csv,17633,134d8488c240fb23e04535db6a94da73006f890c83617eeedaa9b26c9f120015,present


**讀取範圍與CRS／編碼**

,path,rows,encoding,strict,byte_roundtrip,delimiter,invalid_byte_count,invalid_bytes
0,data/raw/臺北捷運車站出入口座標_1150804.csv,390,cp950,True,True,"','",0,[]


### Sample

各實際讀取表前十列；全國事故為臺北篩選後第一當事人，不代表全國前十列。OSM為bbox空間子集。

In [4]:
for source,frame in frames.items():
    show(source,frame.drop(columns='geometry',errors='ignore').head(10),block=10)

**data/raw/臺北捷運車站出入口座標_1150804.csv**

,項次,出入口名稱,出入口編號,經度,緯度,是否為無障礙用,_source_row,_source_path,_decoding_valid
0,1,頂埔站出口1,1,121.4183364,24.9593267,是,1,data/raw/臺北捷運車站出入口座標_1150804.csv,True
1,2,頂埔站出口2,2,121.4183575,24.95894725,否,2,data/raw/臺北捷運車站出入口座標_1150804.csv,True
2,3,頂埔站出口3,3,121.4192802,24.95950272,是,3,data/raw/臺北捷運車站出入口座標_1150804.csv,True
3,4,頂埔站出口4,4,121.4199687,24.96025528,是,4,data/raw/臺北捷運車站出入口座標_1150804.csv,True
4,5,松山機場站出口1,1,121.5520427,25.0636308,是,5,data/raw/臺北捷運車站出入口座標_1150804.csv,True
5,6,松山機場站出口2,2,121.5509869,25.0635351,是,6,data/raw/臺北捷運車站出入口座標_1150804.csv,True
6,7,松山機場站出口3,3,121.5518415,25.0631707,是,7,data/raw/臺北捷運車站出入口座標_1150804.csv,True
7,8,中山國中站出口,0,121.5439794,25.06082643,是,8,data/raw/臺北捷運車站出入口座標_1150804.csv,True
8,9,忠孝復興站出口1,1,121.543143,25.041797,否,9,data/raw/臺北捷運車站出入口座標_1150804.csv,True
9,10,忠孝復興站出口2,2,121.543517,25.041206,是,10,data/raw/臺北捷運車站出入口座標_1150804.csv,True


### Summary Stats

ZIP逐成員CRC、配套、原始WKT、全部欄型態／缺失／定義、重複與數值／類別／時間診斷。保留原值及bytes。

In [5]:
_ = ctx.raw_report(FAMILY,crc=True)

,source_metadata_and_limits
title,臺北捷運出入口
provider,臺北大眾捷運股份有限公司
url,https://data.taipei/dataset/detail?id=cfa4778c-62c1-497b-b704-756231de348b
publication,資源2026-08-04
period,1150804檔名及更新快照，無逐出入口設置／撤除日期
limitation,固定設施快照，不證明研究年存在；站名與入口分開、保留跨市點，直線緩衝非實際步行可及性
license,政府資料開放授權條款第1版；依官方頁，來源與版本須保留


,family,path,bytes,sha256,status
0,mrt,data/raw/臺北捷運車站出入口座標_1150804.csv,17633,134d8488c240fb23e04535db6a94da73006f890c83617eeedaa9b26c9f120015,present


**實際讀取範圍／編碼／CRS／全國或地區計數**

,path,rows,encoding,strict,byte_roundtrip,delimiter,invalid_byte_count,invalid_bytes
0,data/raw/臺北捷運車站出入口座標_1150804.csv,390,cp950,True,True,"','",0,[]


### data/raw/臺北捷運車站出入口座標_1150804.csv

**實際前十列；全國資料表為篩選後臺北第一當事人，非全國母體**

... (5 outputs left out)

**原字串座標解析診斷與摘要；原欄未改**

,index,count,mean,std,min,25%,50%,75%,max
0,經度,390.0,121.522872,0.044107,121.41077,121.501129,121.525282,121.549891,121.618263
1,緯度,390.0,25.044028,0.036895,24.95784,25.028804,25.041786,25.059163,25.168266


,field,missing_or_unparseable,zero
0,經度,0,0
1,緯度,0,0


原始屬性完整列重複（不含診斷追溯欄） 0


本文件只確認資料身分及可讀性；語義／日期不足者限制用途，不強行納入正式Dataset B3。人類驗收仍待完成。

In [6]:
for item in INPUTS:ctx.file(item['path'])
print('原始容器hash重核通過；正式分析資料輸出0。')

原始容器hash重核通過；正式分析資料輸出0。
